# H₄ lineal con VQE: curva de energía, $R_e$ y $D_e$

In [ ]:
# Instalación de dependencias (versiones fijadas con las que desarrollamos)
%pip install -q qiskit==2.6.2 qiskit-aer==0.18.4 qiskit-nature==0.8.0 pyscf==2.15.1

In [ ]:
import time
_notebook_t0 = time.time()

In [ ]:
# ---- src/hamiltonian.py ----
# Owner: Gonza. External libraries only, never `from src.x import ...` (see docs/task_split.md).
"""Chemistry of the linear H4 chain (STO-3G): from the distance d to the qubit Hamiltonian.

Agreed interface (docs/task_split.md):

    build_problem(d) -> dict with the Hamiltonian, the HF bitstring, the reference
    energies (HF, CCSD, FCI), the CCSD t1/t2 amplitudes and the N and S^2 operators.

Conventions (the same ones used by the Quantum Chemistry workshop and by qiskit-nature
with the Jordan-Wigner mapper):
  - 4 spatial orbitals -> 8 spin-orbitals -> 8 qubits.
  - Qubits 0..3 = alpha spin-orbitals, qubits 4..7 = beta spin-orbitals.
  - Molecular orbitals are sorted by Hartree-Fock energy.
  - The Hamiltonian ALREADY includes the nuclear repulsion.
"""
import numpy as np
from pyscf import gto, scf, mcscf, ao2mo, cc
from qiskit.quantum_info import SparsePauliOp

N_HYDROGENS = 4

# Grid in Å: denser near the minimum (~0.9-1.0 Å) and in the stretched region >= 1.8 Å.
# It includes exactly 1.8 and 3.0.
DISTANCES = [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.7, 3.0]


# --------------------------------------------------------------------------
# Jordan-Wigner
# --------------------------------------------------------------------------
def _annihilation(p, n_qubits):
    """Annihilation operator of spin-orbital p (Jordan-Wigner)."""
    return SparsePauliOp.from_sparse_list(
        [("Z" * p + "X", list(range(p + 1)), 0.5),
         ("Z" * p + "Y", list(range(p + 1)), 0.5j)],
        n_qubits,
    )


def _qubit_hamiltonian(h1, h2, constant, n_orb):
    """H = constant + sum h_pq a†_p a_q + 1/2 sum g_pqrs a†_p a†_r a_s a_q (chemists' notation)."""
    n_qubits = 2 * n_orb
    a = [_annihilation(p, n_qubits) for p in range(n_qubits)]
    a_dag = [op.adjoint() for op in a]

    # hop[s][p][q] moves an electron with spin s from orbital q to orbital p
    hop = [[[a_dag[s * n_orb + p].dot(a[s * n_orb + q]).simplify()
             for q in range(n_orb)] for p in range(n_orb)] for s in range(2)]

    terms = [SparsePauliOp("I" * n_qubits, constant)]
    for s in range(2):
        for p in range(n_orb):
            for q in range(n_orb):
                terms.append(h1[p, q] * hop[s][p][q])

    for s in range(2):
        for t in range(2):
            for p in range(n_orb):
                for q in range(n_orb):
                    second_hop = SparsePauliOp.sum(
                        [h2[p, q, r, u] * hop[t][r][u] for r in range(n_orb) for u in range(n_orb)])
                    terms.append(0.5 * hop[s][p][q].dot(second_hop))
                    if s == t:
                        # correction for the operator ordering (same spin)
                        for u in range(n_orb):
                            terms.append(-0.5 * h2[p, q, q, u] * hop[s][p][u])
    return SparsePauliOp.sum(terms).simplify(atol=1e-10)


def number_operator(n_orb):
    """N = sum of n_p = (I - Z_p)/2 over the 2*n_orb qubits."""
    n = 2 * n_orb
    terms = [SparsePauliOp("I" * n, 0.5 * n)]
    for p in range(n):
        terms.append(SparsePauliOp.from_sparse_list([("Z", [p], -0.5)], n))
    return SparsePauliOp.sum(terms).simplify()


def s2_operator(n_orb):
    """S^2 = S- S+ + Sz (Sz + 1), with alpha on qubits 0..n_orb-1 and beta on the next ones."""
    n = 2 * n_orb
    a = [_annihilation(p, n) for p in range(n)]
    a_dag = [op.adjoint() for op in a]
    s_plus = SparsePauliOp.sum([a_dag[p].dot(a[n_orb + p]) for p in range(n_orb)])
    s_minus = s_plus.adjoint()
    sz = 0.5 * SparsePauliOp.sum(
        [a_dag[p].dot(a[p]) - a_dag[n_orb + p].dot(a[n_orb + p]) for p in range(n_orb)])
    return (s_minus.dot(s_plus) + sz.dot(sz) + sz).simplify(atol=1e-10)


# --------------------------------------------------------------------------
# Full problem
# --------------------------------------------------------------------------
def geometry(d, n_h=N_HYDROGENS):
    """Straight chain of n_h hydrogens separated by d (Å), along the z axis."""
    return "; ".join(f"H 0 0 {i * d:.10f}" for i in range(n_h))


def build_problem(d: float) -> dict:
    """Linear H4, STO-3G, separation d (Å)."""
    mol = gto.M(atom=geometry(d), basis="sto-3g", verbose=0)
    hf = scf.RHF(mol)
    hf.conv_tol = 1e-12
    hf.run()
    if not hf.converged:
        raise RuntimeError(f"Hartree-Fock did not converge for d={d}")

    n_orb = mol.nao_nr()
    n_elec = mol.nelectron
    n_alpha = n_beta = n_elec // 2

    # Integrals in the molecular-orbital basis (all orbitals active)
    active = mcscf.CASCI(hf, n_orb, n_elec)
    h1, constant = active.get_h1cas()           # constant includes the nuclear repulsion
    h2 = ao2mo.restore(1, active.get_h2cas(), n_orb)
    e_fci = float(active.kernel()[0])

    ccsd = cc.CCSD(hf)
    ccsd.conv_tol = 1e-10
    ccsd.run()
    if not ccsd.converged:
        raise RuntimeError(f"CCSD did not converge for d={d}")

    # Hartree-Fock state: qubits 0..n_alpha-1 (alpha) and n_orb..n_orb+n_beta-1 (beta) occupied.
    # Qiskit convention: qubit 0 is the rightmost character.
    occupied = [1 if (q < n_alpha or n_orb <= q < n_orb + n_beta) else 0 for q in range(2 * n_orb)]
    hf_bitstring = "".join(str(b) for b in occupied[::-1])

    return {
        "d": d,
        "hamiltonian": _qubit_hamiltonian(h1, h2, constant, n_orb),
        "n_qubits": 2 * n_orb,
        "hf_bitstring": hf_bitstring,
        "e_hf": float(hf.e_tot),
        "e_ccsd": float(ccsd.e_tot),
        "e_fci": e_fci,
        "ccsd_t1": np.array(ccsd.t1),   # (n_occupied, n_virtual), spatial orbitals
        "ccsd_t2": np.array(ccsd.t2),   # (o, o, v, v), spatial orbitals
        "number_op": number_operator(n_orb),
        "s2_op": s2_operator(n_orb),
    }


# --------------------------------------------------------------------------
# Validation
# --------------------------------------------------------------------------
def _hf_state(problem):
    """HF state as a statevector (a single computational basis state)."""
    v = np.zeros(2 ** problem["n_qubits"], dtype=complex)
    v[int(problem["hf_bitstring"], 2)] = 1.0
    return v


def validate_problem(problem: dict, penalty: float = 10.0) -> dict:
    """Lowest eigenvalue of H restricted to N=4 and S^2=0, compared with PySCF's FCI.

    A penalty on (N - 4)^2 and on S^2 is added to isolate the singlet 4-electron sector.
    Returns the error |E_min - e_fci| and the expectation values of N and S^2 in that state.
    """
    H = problem["hamiltonian"].to_matrix()
    N = problem["number_op"].to_matrix()
    S2 = problem["s2_op"].to_matrix()
    n_elec = sum(int(b) for b in problem["hf_bitstring"])
    identity = np.eye(H.shape[0])
    penalized = H + penalty * (N - n_elec * identity) @ (N - n_elec * identity) + penalty * S2
    _, vectors = np.linalg.eigh(penalized)
    psi = vectors[:, 0]
    e_min = float(np.real(psi.conj() @ H @ psi))
    hf = _hf_state(problem)
    return {
        "e_min": e_min,
        "error": abs(e_min - problem["e_fci"]),
        "N": float(np.real(psi.conj() @ N @ psi)),
        "S2": float(np.real(psi.conj() @ S2 @ psi)),
        "e_hf_check": float(np.real(hf.conj() @ H @ hf)),
    }

In [ ]:
# ---- src/ansatz.py ----
# Convención de qubits de Jordan-Wigner,
# qubits 0..norb-1 = spin-orbitales alfa, norb..2norb-1 = beta.
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Z2Symmetries
from qiskit.transpiler import CouplingMap
from qiskit_nature.second_q.circuit.library import PUCCD, UCC, UCCSD, HartreeFock
from qiskit_nature.second_q.mappers import JordanWignerMapper, TaperedQubitMapper


def _occupied(problem):
    """Qubits ocupados en HF. Convención Qiskit (0 = último qubit)"""
    return [i for i, b in enumerate(problem["hf_bitstring"][::-1]) if b == "1"]


def _hf_circuit(problem):
    """Circuito de Hartree-Fock"""
    qc = QuantumCircuit(problem["n_qubits"])
    for i in _occupied(problem):
        qc.x(i)
    return qc


def _orbitals(problem):
    """(orbitales espaciales, (e alfa, e beta), mapper). Con tapering vienen guardados en problem."""
    if "mapper" in problem:
        return problem["num_spatial_orbitals"], problem["num_particles"], problem["mapper"]
    norb = problem["n_qubits"] // 2
    n_alpha = sum(q < norb for q in _occupied(problem))
    return norb, (n_alpha, len(_occupied(problem)) - n_alpha), JordanWignerMapper()


def taper_problem(problem: dict) -> dict:
    """Problema con menos qubits usando las simetrías Z2 del hamiltoniano (H4: de 8 a 5 qubits).

    Se queda con el sector que tiene el estado HF."""
    H = problem["hamiltonian"]
    norb, particles, mapper = _orbitals(problem)
    z = Z2Symmetries.find_z2_symmetries(H)
    hf = problem["hf_bitstring"][::-1]
    # Las simetrías son cadenas de Z. El autovalor sobre HF es (-1)^(cantidad de Z sobre qubits ocupados)
    sector = [(-1) ** sum(c == "Z" and hf[q] == "1" for q, c in enumerate(s.to_label()[::-1])) for s in z.symmetries]
    z2 = Z2Symmetries(z.symmetries, z.sq_paulis, z.sq_list, sector)
    tapered_mapper = TaperedQubitMapper(mapper, z2)

    # Estado HF en los qubits que quedan
    hf_state = Statevector(HartreeFock(norb, particles, tapered_mapper))
    n = hf_state.num_qubits
    tapered = dict(problem, hamiltonian=z2.taper(H), n_qubits=n, mapper=tapered_mapper,
                   num_spatial_orbitals=norb, num_particles=particles,
                   hf_bitstring=format(int(np.argmax(abs(hf_state.data))), f"0{n}b"))
    for key in ("number_op", "s2_op"):  # N y S² conmutan con las simetrías
        if key in problem:
            tapered[key] = z2.taper(problem[key])
    return tapered


def _ucc(problem: dict, kind: str, k: int = 1, excitations=None):
    """Ansatz parte de HF."""
    hf = _hf_circuit(problem)  # el circuito inicia en el estado de Hartree-Fock
    norb, particles, mapper = _orbitals(problem)
    args = (norb, particles, mapper)

    if kind == "uccsd":
        return UCCSD(*args, reps=k, initial_state=hf)
    if kind == "uccgsd":
        # Saltos entre cualquier par de orbitales (incluye ocupados a vacíos)
        return UCCSD(*args, reps=k, generalized=True, initial_state=hf)
    if kind == "kupccgsd":
        # Dobles en pares (alfa y beta juntos) y simples, entre todos los orbitales, repetido k veces
        return PUCCD(*args, reps=k, include_singles=(True, True), generalized=True, initial_state=hf)
    if kind == "custom":
        # Solo los saltos de `excitations`, en ese orden (lo usa la poda)
        return UCC(norb, particles, excitations=lambda num_spatial_orbitals, num_particles: excitations, qubit_mapper=mapper, initial_state=hf)
    raise ValueError(f"ansatz inválido: {kind!r}")


def excitation_list(problem: dict, kind: str, k: int = 1) -> list:
    """Saltos (desde, hacia) de un ansatz, en el mismo orden que sus parámetros θ."""
    return _ucc(problem, kind, k).excitation_list


def build_ansatz(problem: dict, kind: str, k: int = 1, excitations=None) -> QuantumCircuit:
    """kind: 'hf' (solo HF), 'uccsd', 'uccgsd' (UCCSD generalizado), 'kupccgsd' o 'custom' (requiere `excitations`).
    k = cuántas veces se repite el bloque (no aplica a 'hf' ni a 'custom')."""
    if kind == "hf":
        return _hf_circuit(problem)
    ansatz = _ucc(problem, kind, k, excitations)

    # Se descompone en compuertas para simular rápido.
    return transpile(ansatz, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1, seed_transpiler=0)


def count_cz(circuit: QuantumCircuit) -> int:
    """Transpila a ['cz','sx','x','rz'] con conectividad en línea y cuenta cz (costo de simulación)."""
    t = transpile(circuit, basis_gates=["cz", "sx", "x", "rz"],
                  coupling_map=CouplingMap.from_line(circuit.num_qubits),
                  optimization_level=1, seed_transpiler=0)
    return t.count_ops().get("cz", 0)

In [ ]:
# ---- src/vqe.py ----
import numpy as np
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator
import json, os


def optimize(problem: dict, ansatz: "QuantumCircuit", theta0=None, method: str = "L-BFGS-B") -> dict:
    """Minimiza <H> con statevector exacto.
    Devuelve {"theta": ndarray, "e_exact": float, "n_iters": int}.
    """
    est = StatevectorEstimator()
    H = problem["hamiltonian"]
    x0 = np.zeros(ansatz.num_parameters) if theta0 is None else np.asarray(theta0, dtype=float)

    def energy(theta):
        return float(est.run([(ansatz, H, theta)]).result()[0].data.evs)

    res = minimize(energy, x0, method=method, tol=1e-12)
    return {"theta": res.x, "e_exact": float(res.fun), "n_iters": int(res.nfev)}


def prune(problem: dict, kind: str, theta, tol: float = 1e-3) -> dict:
    """Poda saca del ansatz los saltos con |θ| <= tol y vuelve a optimizar el resto.
    Devuelve lo mismo que optimize, "ansatz" (circuito podado) y "excitations" (saltos que quedaron).
    """
    theta = np.asarray(theta, dtype=float)
    keep = np.abs(theta) > tol
    excitations = [e for e, k in zip(excitation_list(problem, kind), keep) if k]
    ansatz = build_ansatz(problem, "custom", excitations=excitations)
    return {"ansatz": ansatz, "excitations": excitations, **optimize(problem, ansatz, theta0=theta[keep])}


def sweep(kind: str, distances, out_dir: str = "results", k: int = 1, taper: bool = False) -> list:
    """Barrido de la curva con warm start (θ de cada d son el inicio del siguiente).

    kind: cualquiera de build_ansatz.
    taper: usa el problema con menos qubits (taper_problem).
    Guarda results/<kind>[k][_tap]_d<d>.json. Si ya existe, lo reutiliza.
    """
    os.makedirs(out_dir, exist_ok=True)
    name = (f"{kind}{k}" if k > 1 or kind == "kupccgsd" else kind) + ("_tap" if taper else "")
    theta, rows = None, []
    for d in sorted(distances):
        path = f"{out_dir}/{name}_d{d}.json"
        if os.path.exists(path):
            row = json.load(open(path))
        else:
            problem = build_problem(d)
            if taper:
                problem = taper_problem(problem)
            ansatz = build_ansatz(problem, kind, k)
            r = optimize(problem, ansatz, theta0=theta)
            row = {
                "d": d,
                "kind": name,
                "theta": r["theta"].tolist(),
                "e_exact": r["e_exact"],
                "n_iters": r["n_iters"],
                "cz": count_cz(ansatz),
                "n_qubits": problem["n_qubits"],
                "e_hf": problem["e_hf"],
                "e_ccsd": problem["e_ccsd"],
                "e_fci": problem["e_fci"],
            }
            json.dump(row, open(path, "w"), indent=1)
        theta = np.array(row["theta"]) if row["theta"] else None
        rows.append(row)
        print(
            f"d={d}: error {1000 * (row['e_exact'] - row['e_fci']):.4f} mHa (CCSD {1000 * (row['e_ccsd'] - row['e_fci']):.4f}), "
            f"cz {row['cz']}, parámetros {len(row['theta'])}, evaluaciones {row['n_iters']}",
            flush=True,
        )
    return rows

In [ ]:
# ---- src/measurement.py ----
# Dueño: Maxi. Solo librerías externas, nunca `from src.x import ...`.
"""Shot-based estimation of expectation values with qiskit_aer's SamplerV2.

Public API (interface in docs/task_split.md):
    measure_energy(circuit, theta, hamiltonian, total_shots) -> {"energy", "sigma", "shots"}

Extras:
    measure_expectation   same thing for any SparsePauliOp (N, S², H², ...)
    validate_sigma        repeat the measurement with different seeds and compare
                          the spread of the energies with the reported sigma
    shots_for_sigma       shots needed to reach a target sigma (sigma ~ 1/sqrt(shots))
    compare_allocations   uniform vs |c_k|-weighted vs adaptive shot allocation
    measure_reliability   <N>, <S²> and Var(H) = <H²> - <H>², measured with shots

Shot allocation strategies (`allocation=`):
    "uniform"   same number of shots for every qubit-wise commuting group
    "weighted"  shots proportional to sum_k |c_k| over the group
    "adaptive"  a pilot run (uniform) estimates the per-shot std sigma_g of every
                group, and the remaining shots go proportional to sigma_g, which
                minimizes sum_g sigma_g² / N_g for a fixed total (Neyman allocation)
"""
import numpy as np
from qiskit import transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2

ALLOCATIONS = ("uniform", "weighted", "adaptive")
MIN_SHOTS_PER_GROUP = 2   # at least 2 shots so that the sample variance is defined
PILOT_FRACTION = 0.1      # fraction of the budget spent on the pilot run in "adaptive"


# --------------------------------------------------------------------------
# Building blocks
# --------------------------------------------------------------------------
def _split_identity(operator):
    """Return (constant, operator without the identity term). The constant needs no shots."""
    operator = operator.simplify(atol=1e-12)
    labels = operator.paulis.to_labels()
    identity = "I" * operator.num_qubits
    constant = float(np.real(sum(c for l, c in zip(labels, operator.coeffs) if l == identity)))
    keep = [i for i, l in enumerate(labels) if l != identity]
    rest = operator[keep] if keep else None
    return constant, rest


def _group_basis(group):
    """Basis per qubit ('I', 'X', 'Y' or 'Z') shared by all terms of a qubit-wise commuting group."""
    n = group.num_qubits
    basis = ["I"] * n
    for label in group.paulis.to_labels():
        for q in range(n):
            p = label[n - 1 - q]  # Qiskit labels: qubit 0 is the rightmost character
            if p != "I":
                basis[q] = p
    return basis


def _measurement_circuit(state_circuit, basis):
    """Append the basis-change rotations and a measurement of every qubit."""
    qc = state_circuit.copy()
    qc.barrier()
    for q, p in enumerate(basis):
        if p == "X":
            qc.h(q)
        elif p == "Y":
            qc.sdg(q)
            qc.h(q)
    qc.measure_all()
    return qc


def _per_shot_values(counts, group):
    """Return (values, weights): value of sum_k c_k P_k for each distinct bitstring, and its count.

    A Pauli string measured in its eigenbasis gives the parity of the bits on its support:
    +1 for an even number of ones, -1 for an odd number.
    """
    coeffs = np.real(group.coeffs)
    support = group.paulis.x | group.paulis.z                 # (n_terms, n_qubits), qubit 0 first
    bitstrings = list(counts)
    # bitstring[-1] is qubit 0, so reverse to get columns in qubit order
    bits = np.array([[int(b) for b in s.replace(" ", "")[::-1]] for s in bitstrings], dtype=int)
    parity = (bits @ support.T.astype(int)) % 2                # (n_outcomes, n_terms)
    values = (1 - 2 * parity) @ coeffs
    weights = np.array([counts[s] for s in bitstrings], dtype=float)
    return values, weights


def _group_stats(counts, group):
    """Mean and per-shot sample variance of the group's observable."""
    values, weights = _per_shot_values(counts, group)
    n = weights.sum()
    mean = np.sum(weights * values) / n
    var = np.sum(weights * (values - mean) ** 2) / max(n - 1, 1)
    return mean, var, n


def _distribute(total, scores):
    """Split `total` shots proportionally to `scores` (largest remainder), at least MIN_SHOTS_PER_GROUP each."""
    k = len(scores)
    scores = np.asarray(scores, dtype=float)
    floor = MIN_SHOTS_PER_GROUP * k
    if total <= floor:
        return np.full(k, max(MIN_SHOTS_PER_GROUP, total // k), dtype=int)
    if scores.sum() <= 0:
        scores = np.ones(k)
    extra = total - floor
    raw = extra * scores / scores.sum()
    shots = np.floor(raw).astype(int)
    remainder = extra - shots.sum()
    shots[np.argsort(raw - shots)[::-1][:remainder]] += 1
    return shots + MIN_SHOTS_PER_GROUP


def _run(sampler, circuits, shots):
    """One SamplerV2 job with a different number of shots per circuit; returns a list of counts."""
    pubs = [(qc, None, int(s)) for qc, s in zip(circuits, shots)]
    result = sampler.run(pubs).result()
    return [r.data.meas.get_counts() for r in result]


# --------------------------------------------------------------------------
# Measurement
# --------------------------------------------------------------------------
def measure_expectation(circuit, theta, operator, total_shots: int,
                        allocation: str = "weighted", seed=None) -> dict:
    """Estimate <operator> with `total_shots` shots spread over qubit-wise commuting groups.

    The sigma accounts for the covariances inside each group exactly, because the
    per-shot value of the whole group (sum_k c_k P_k on one bitstring) is what gets
    averaged. Different groups are independent circuits, so their variances add.

    Returns {"energy", "sigma", "shots", "n_groups", "allocation", "groups"}; "groups"
    has the shots, mean and per-shot std of every group.
    """
    if allocation not in ALLOCATIONS:
        raise ValueError(f"allocation must be one of {ALLOCATIONS}, got {allocation!r}")

    qc = circuit.assign_parameters(theta) if circuit.num_parameters > 0 else circuit
    constant, rest = _split_identity(operator)
    if rest is None:
        return {"energy": constant, "sigma": 0.0, "shots": 0, "n_groups": 0,
                "allocation": allocation, "groups": []}

    groups = rest.group_commuting(qubit_wise=True)
    backend = AerSimulator()
    circuits = transpile([_measurement_circuit(qc, _group_basis(g)) for g in groups], backend)
    # Aer seeds the i-th circuit of a job with seed_simulator + i, so nearby user seeds
    # would reuse random streams across calls. Derive well-separated job seeds instead.
    job_seeds = np.random.SeedSequence(seed).generate_state(2) if seed is not None else None

    def sampler_for(job):
        if job_seeds is None:
            return SamplerV2()
        return SamplerV2(options={"backend_options": {"seed_simulator": int(job_seeds[job])}})

    sampler = sampler_for(0)

    if allocation == "uniform":
        shots = _distribute(total_shots, np.ones(len(groups)))
        counts = _run(sampler, circuits, shots)
    elif allocation == "weighted":
        shots = _distribute(total_shots, [np.sum(np.abs(g.coeffs)) for g in groups])
        counts = _run(sampler, circuits, shots)
    else:  # adaptive
        pilot = _distribute(int(PILOT_FRACTION * total_shots), np.ones(len(groups)))
        pilot_counts = _run(sampler, circuits, pilot)
        stds = np.array([np.sqrt(_group_stats(c, g)[1]) for c, g in zip(pilot_counts, groups)])
        # if a group gave zero spread in the pilot, keep a small weight so it is not starved
        stds = np.maximum(stds, 1e-3 * max(stds.max(), 1e-12))
        # Neyman target for the total budget, minus what each group already got in the pilot
        target = _distribute(total_shots, stds)
        main = np.maximum(target - pilot, 0)
        main_counts = _run(sampler_for(1), circuits, np.maximum(main, 1))
        counts = []
        for pc, mc, m in zip(pilot_counts, main_counts, main):
            merged = dict(pc)
            if m > 0:
                for k, v in mc.items():
                    merged[k] = merged.get(k, 0) + v
            counts.append(merged)
        shots = pilot + main

    energy = constant
    variance = 0.0
    details = []
    for group, c in zip(groups, counts):
        mean, var, n = _group_stats(c, group)
        energy += mean
        variance += var / n
        details.append({"n_terms": len(group), "shots": int(n),
                        "mean": float(mean), "std_per_shot": float(np.sqrt(var))})

    return {
        "energy": float(energy),
        "sigma": float(np.sqrt(variance)),
        "shots": int(sum(d["shots"] for d in details)),
        "n_groups": len(groups),
        "allocation": allocation,
        "groups": details,
    }


def measure_energy(circuit, theta, hamiltonian, total_shots: int,
                   allocation: str = "weighted", seed=None) -> dict:
    """Devuelve {"energy": float, "sigma": float, "shots": int}.

    Groups the Pauli terms into qubit-wise commuting sets, measures one circuit
    per group (shots split according to `allocation`) and returns the energy
    estimate with its statistical standard error. The identity term is added
    exactly and costs no shots.
    """
    r = measure_expectation(circuit, theta, hamiltonian, total_shots, allocation, seed)
    return {"energy": r["energy"], "sigma": r["sigma"], "shots": r["shots"]}


# --------------------------------------------------------------------------
# Validation of sigma and shot budget
# --------------------------------------------------------------------------
def validate_sigma(circuit, theta, hamiltonian, total_shots: int, n_repeats: int = 30,
                   allocation: str = "weighted", seed0: int = 0) -> dict:
    """Repeat measure_energy `n_repeats` times with seeds seed0, seed0+1, ...

    If sigma is right, the standard deviation of the energies ("std_energies")
    matches the reported sigma ("mean_sigma") within the sampling error of a std
    estimated from n samples, about std / sqrt(2 (n - 1)).
    """
    runs = [measure_energy(circuit, theta, hamiltonian, total_shots, allocation, seed=seed0 + i)
            for i in range(n_repeats)]
    energies = np.array([r["energy"] for r in runs])
    sigmas = np.array([r["sigma"] for r in runs])
    std = float(np.std(energies, ddof=1))
    return {
        "allocation": allocation,
        "total_shots": int(runs[0]["shots"]),
        "n_repeats": n_repeats,
        "mean_energy": float(energies.mean()),
        "std_energies": std,
        "std_energies_error": std / np.sqrt(2 * (n_repeats - 1)),
        "mean_sigma": float(sigmas.mean()),
        "ratio": std / float(sigmas.mean()),
        "energies": energies,
        "sigmas": sigmas,
    }


def shots_for_sigma(sigma: float, shots: int, target: float = 1e-3) -> int:
    """Shots needed to reach `target` given that `shots` gave `sigma` (sigma ~ 1/sqrt(shots))."""
    return int(np.ceil(shots * (sigma / target) ** 2))


def compare_allocations(circuit, theta, hamiltonian, total_shots: int, n_repeats: int = 30,
                        target: float = 1e-3, allocations=ALLOCATIONS, seed0: int = 0) -> list:
    """For every allocation: reported sigma, empirical std over repeats, and shots for sigma < target."""
    rows = []
    for allocation in allocations:
        v = validate_sigma(circuit, theta, hamiltonian, total_shots, n_repeats, allocation, seed0)
        rows.append({
            "allocation": allocation,
            "shots": v["total_shots"],
            "mean_energy": v["mean_energy"],
            "mean_sigma": v["mean_sigma"],
            "std_energies": v["std_energies"],
            "ratio": v["ratio"],
            "shots_for_target": shots_for_sigma(v["mean_sigma"], v["total_shots"], target),
        })
    return rows


# --------------------------------------------------------------------------
# Reliability indicators (theory notes, section 11)
# --------------------------------------------------------------------------
def measure_reliability(circuit, theta, hamiltonian, number_op, s2_op, total_shots: int,
                        allocation: str = "weighted", seed=None) -> dict:
    """Measure <N>, <S²> and Var(H) with shots, using the same code as the energy.

    Var(H) = <(H - a)²> - (<H> - a)². Shifting by a = measured <H> keeps the
    coefficients of (H - a)² small, so its estimate is much less noisy than
    measuring <H²> and <H>² separately; the second term is then ~sigma_E² and
    is dropped. Var(H) = 0 if and only if the state is an eigenstate.
    """
    s = None if seed is None else int(seed)
    energy = measure_expectation(circuit, theta, hamiltonian, total_shots, allocation, s)
    a = energy["energy"]
    n_qubits = hamiltonian.num_qubits
    shifted = (hamiltonian - SparsePauliOp("I" * n_qubits, a)).simplify(atol=1e-12)
    h2 = shifted.dot(shifted).simplify(atol=1e-12)
    var = measure_expectation(circuit, theta, h2, total_shots, allocation,
                              None if s is None else s + 10)
    n = measure_expectation(circuit, theta, number_op, total_shots, allocation,
                            None if s is None else s + 20)
    s2 = measure_expectation(circuit, theta, s2_op, total_shots, allocation,
                             None if s is None else s + 30)
    return {
        "energy": a, "energy_sigma": energy["sigma"],
        "variance": var["energy"], "variance_sigma": var["sigma"],
        "N": n["energy"], "N_sigma": n["sigma"],
        "S2": s2["energy"], "S2_sigma": s2["sigma"],
        "shots": energy["shots"] + var["shots"] + n["shots"] + s2["shots"],
    }

In [ ]:
# ---- src/analysis.py ----
# Dueño: Maxi. Lee results/<kind>_d<d>.json y arma tabla, gráficos, R_e y D_e.
# Solo librerías externas, nunca `from src.x import ...`.
"""Results bookkeeping and the final analysis of the potential energy curve.

A *record* is one point of the curve, a JSON-serializable dict:
    {"kind", "d", "energy", "sigma", "shots", "cz", "theta",
     "e_hf", "e_ccsd", "e_fci", ...optional extras (allocation, reliability...)}

Pipeline:
    make_record(...) -> save_record(...)            one point -> results/<kind>_d<d>.json
    load_records(kind)                              all points of one ansatz, sorted by d
    results_table(records)                          pandas table for the notebook
    plot_curve(records), plot_errors(records)       figures
    fit_equilibrium(d, E, sigma)                    R_e and E(R_e) with bootstrap errors
    dissociation_energy(records)                    D_e = E(3.0 Å) - E(R_e)
    beats_ccsd(records)                             |E - E_FCI| + sigma < |E_CCSD - E_FCI| for d >= 1.8
    summary(records)                                all of the above for VQE, CCSD and FCI
"""
import glob
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar

RESULTS_DIR = "results"
HARTREE_TO_MHA = 1000.0
CHEMICAL_ACCURACY = 1.6e-3   # Ha
D_STRETCHED = 1.8            # Å, from here on we have to beat CCSD
D_DISSOCIATED = 3.0          # Å, atoms considered separated (D_e definition in CONSIGNA.md)


# --------------------------------------------------------------------------
# Records
# --------------------------------------------------------------------------
def make_record(kind: str, problem: dict, theta, measurement: dict, cz: int, **extra) -> dict:
    """Build a record from build_problem's dict, the optimal theta and measure_energy's result."""
    record = {
        "kind": kind,
        "d": float(problem["d"]),
        "energy": float(measurement["energy"]),
        "sigma": float(measurement["sigma"]),
        "shots": int(measurement["shots"]),
        "cz": int(cz),
        "theta": [float(t) for t in np.atleast_1d(theta)] if theta is not None else [],
        "e_hf": float(problem["e_hf"]),
        "e_ccsd": float(problem["e_ccsd"]),
        "e_fci": float(problem["e_fci"]),
    }
    record.update(_to_jsonable(extra))
    return record


def _to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def record_path(kind: str, d: float, results_dir: str = RESULTS_DIR) -> str:
    return os.path.join(results_dir, f"{kind}_d{d:.2f}.json")


def save_record(record: dict, results_dir: str = RESULTS_DIR) -> str:
    os.makedirs(results_dir, exist_ok=True)
    path = record_path(record["kind"], record["d"], results_dir)
    with open(path, "w") as f:
        json.dump(record, f, indent=2)
    return path


def load_records(kind: str, results_dir: str = RESULTS_DIR) -> list:
    records = []
    for path in glob.glob(os.path.join(results_dir, f"{kind}_d*.json")):
        with open(path) as f:
            records.append(json.load(f))
    return sorted(records, key=lambda r: r["d"])


def _arrays(records):
    records = sorted(records, key=lambda r: r["d"])
    get = lambda key: np.array([r[key] for r in records], dtype=float)
    return records, get


# --------------------------------------------------------------------------
# Table
# --------------------------------------------------------------------------
def results_table(records) -> pd.DataFrame:
    """One row per distance: E ± sigma, shots, cz, references and errors in mHa."""
    rows = []
    for r in sorted(records, key=lambda r: r["d"]):
        err = abs(r["energy"] - r["e_fci"])
        err_ccsd = abs(r["e_ccsd"] - r["e_fci"])
        stretched = r["d"] >= D_STRETCHED - 1e-9
        rows.append({
            "d (Å)": r["d"],
            "E (Ha)": r["energy"],
            "σ (mHa)": r["sigma"] * HARTREE_TO_MHA,
            "shots": r["shots"],
            "cz": r["cz"],
            "E_CCSD (Ha)": r["e_ccsd"],
            "E_FCI (Ha)": r["e_fci"],
            "|E-E_FCI| (mHa)": err * HARTREE_TO_MHA,
            "|E_CCSD-E_FCI| (mHa)": err_ccsd * HARTREE_TO_MHA,
            "le gana a CCSD": bool(err + r["sigma"] < err_ccsd) if stretched else None,
            "precisión química": bool(err < CHEMICAL_ACCURACY),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Plots
# --------------------------------------------------------------------------
def plot_curve(records, ax=None, label="VQE"):
    """E(d) with ±sigma error bars, against HF, CCSD and FCI."""
    records, get = _arrays(records)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    d = get("d")
    ax.plot(d, get("e_hf"), ":", color="gray", label="HF")
    ax.plot(d, get("e_ccsd"), "s--", color="tab:orange", ms=4, label="CCSD")
    ax.plot(d, get("e_fci"), "-", color="black", lw=1, label="FCI")
    ax.errorbar(d, get("energy"), yerr=get("sigma"), fmt="o", color="tab:blue",
                ms=4, capsize=3, label=f"{label} (±σ)")
    ax.set_xlabel("d (Å)")
    ax.set_ylabel("E (Ha)")
    ax.set_title("H₄ lineal, STO-3G")
    ax.legend()
    ax.grid(alpha=0.3)
    return ax


def plot_errors(records, ax=None, label="VQE"):
    """|E - E_FCI| in mHa (with sigma) and the CCSD error, log scale, chemical accuracy line."""
    records, get = _arrays(records)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    d = get("d")
    err = np.abs(get("energy") - get("e_fci")) * HARTREE_TO_MHA
    sig = get("sigma") * HARTREE_TO_MHA
    err_ccsd = np.abs(get("e_ccsd") - get("e_fci")) * HARTREE_TO_MHA
    ax.plot(d, err_ccsd, "s--", color="tab:orange", ms=4, label="|E_CCSD − E_FCI|")
    ax.errorbar(d, err, yerr=sig, fmt="o", color="tab:blue", ms=4, capsize=3,
                label=f"|E_{label} − E_FCI| (±σ)")
    ax.plot(d, err + sig, "v", color="tab:blue", ms=4, alpha=0.4, label="|E − E_FCI| + σ")
    ax.axhline(CHEMICAL_ACCURACY * HARTREE_TO_MHA, color="green", ls=":", label="1,6 mHa")
    ax.axvline(D_STRETCHED, color="gray", ls=":", lw=1)
    ax.set_yscale("log")
    ax.set_xlabel("d (Å)")
    ax.set_ylabel("error (mHa)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, which="both")
    return ax


# --------------------------------------------------------------------------
# R_e and D_e
# --------------------------------------------------------------------------
def _minimum(d, e, method, n_points):
    """Location and value of the minimum of the curve through (d, e)."""
    if method == "parabola":
        i = int(np.argmin(e))
        half = n_points // 2
        lo = max(0, min(i - half, len(d) - n_points))
        sel = slice(lo, lo + n_points)
        a, b, c = np.polyfit(d[sel], e[sel], 2)
        if a <= 0:
            return d[i], e[i]
        x = -b / (2 * a)
        return x, c - b ** 2 / (4 * a)
    if method == "spline":
        spline = CubicSpline(d, e)
        i = int(np.argmin(e))
        lo, hi = d[max(i - 1, 0)], d[min(i + 1, len(d) - 1)]
        res = minimize_scalar(spline, bounds=(lo, hi), method="bounded")
        return float(res.x), float(res.fun)
    raise ValueError("method must be 'parabola' or 'spline'")


def fit_equilibrium(d, energies, sigmas=None, method: str = "spline", n_points: int = 5,
                    n_bootstrap: int = 2000, seed: int = 0) -> dict:
    """R_e and E(R_e) from a cubic spline through all points (default), or a parabola
    through the n_points closest to the minimum.

    On the 0.1 Å grid the spline lands within ~0.3 mÅ of the dense-grid FCI minimum,
    while the parabola is off by 10 to 30 mÅ because the curve is not symmetric.

    Errors come from a parametric bootstrap: the energies are resampled as
    E_i + sigma_i * N(0, 1) and the fit is repeated. With sigmas=None (CCSD, FCI)
    the errors are zero. This is only the statistical error, the grid spacing
    also limits the precision of R_e.
    """
    d = np.asarray(d, dtype=float)
    e = np.asarray(energies, dtype=float)
    order = np.argsort(d)
    d, e = d[order], e[order]
    r_e, e_min = _minimum(d, e, method, n_points)
    r_e_sigma = e_min_sigma = 0.0
    if sigmas is not None and np.any(np.asarray(sigmas) > 0):
        s = np.asarray(sigmas, dtype=float)[order]
        rng = np.random.default_rng(seed)
        samples = np.array([_minimum(d, e + s * rng.standard_normal(len(e)), method, n_points)
                            for _ in range(n_bootstrap)])
        r_e_sigma, e_min_sigma = samples.std(axis=0, ddof=1)
    return {"R_e": float(r_e), "R_e_sigma": float(r_e_sigma),
            "E_min": float(e_min), "E_min_sigma": float(e_min_sigma), "method": method}


def dissociation_energy(records, key: str = "energy", method: str = "spline",
                        d_dissociated: float = D_DISSOCIATED, **fit_kwargs) -> dict:
    """D_e = E(3.0 Å) - E(R_e) for the curve stored under `key` ("energy", "e_ccsd" or "e_fci")."""
    records, get = _arrays(records)
    d = get("d")
    e = get(key)
    s = get("sigma") if key == "energy" else None
    j = int(np.argmin(np.abs(d - d_dissociated)))
    if abs(d[j] - d_dissociated) > 1e-6:
        raise ValueError(f"no point at d = {d_dissociated} Å")
    fit = fit_equilibrium(d, e, s, method=method, **fit_kwargs)
    s_far = 0.0 if s is None else s[j]
    return {
        **fit,
        "E_dissociated": float(e[j]),
        "D_e": float(e[j] - fit["E_min"]),
        "D_e_sigma": float(np.hypot(s_far, fit["E_min_sigma"])),
    }


def beats_ccsd(records, d_min: float = D_STRETCHED) -> dict:
    """Check |E - E_FCI| + sigma < |E_CCSD - E_FCI| on every point with d >= d_min."""
    rows = []
    for r in sorted(records, key=lambda r: r["d"]):
        if r["d"] < d_min - 1e-9:
            continue
        lhs = abs(r["energy"] - r["e_fci"]) + r["sigma"]
        rhs = abs(r["e_ccsd"] - r["e_fci"])
        rows.append({"d": r["d"], "lhs_mHa": lhs * HARTREE_TO_MHA, "rhs_mHa": rhs * HARTREE_TO_MHA,
                     "ok": bool(lhs < rhs),
                     "chemical_accuracy": bool(abs(r["energy"] - r["e_fci"]) < CHEMICAL_ACCURACY)})
    return {"all_ok": bool(rows) and all(r["ok"] for r in rows),
            "all_chemical_accuracy": bool(rows) and all(r["chemical_accuracy"] for r in rows),
            "points": rows}


def summary(records, method: str = "spline", **fit_kwargs) -> pd.DataFrame:
    """R_e and D_e for VQE (with errors), CCSD and FCI, side by side."""
    rows = []
    for name, key in (("VQE", "energy"), ("CCSD", "e_ccsd"), ("FCI", "e_fci")):
        r = dissociation_energy(records, key=key, method=method, **fit_kwargs)
        rows.append({"método": name,
                     "R_e (Å)": r["R_e"], "σ R_e (Å)": r["R_e_sigma"],
                     "E(R_e) (Ha)": r["E_min"],
                     "D_e (Ha)": r["D_e"], "σ D_e (mHa)": r["D_e_sigma"] * HARTREE_TO_MHA})
    return pd.DataFrame(rows)


def report(records, method: str = "spline"):
    """Everything the notebook shows at the end: table, figures, R_e/D_e and the CCSD check."""
    table = results_table(records)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    plot_curve(records, axes[0])
    plot_errors(records, axes[1])
    fig.tight_layout()
    plt.close(fig)   # do not auto-display here; show it with out["figure"]
    check = beats_ccsd(records)
    return {"table": table, "summary": summary(records, method=method),
            "beats_ccsd": check, "figure": fig}

In [ ]:
# ---- src/pipeline.py ----
# Dueño: Maxi (integración). Solo librerías externas, nunca `from src.x import ...`.
# Uses build_problem (hamiltonian.py), taper_problem / build_ansatz / excitation_list /
# count_cz (ansatz.py), measure_energy / measure_reliability (measurement.py) and
# make_record (analysis.py) from the shared namespace, like every other notebook cell.
"""From Fede's optimal θ to measured energies: the reported E ± σ of every point.

The θ come from the statevector optimization (allowed by the rules); the energies
reported here are measured with shots through measure_energy (SamplerV2), as the
rules require.

    points = load_points("uccsd_tap")       # {d: {"theta", "e_exact"}} from results/uccsd_tap_d<d>.json
    records = measure_curve("uccsd_tap", points, total_shots=5_000_000)
    out = report(records)

Orbital signs: PySCF returns every molecular orbital up to an arbitrary sign, and
the sign can change between runs (it does at d = 3.0 Å, even inside one process).
Flipping orbital p flips the sign of every excitation amplitude that touches p, so
a θ optimized in one run can give a much higher energy in another. align_theta
undoes this: it tries the 2^n_orb sign patterns on θ and keeps the one whose exact
(statevector) energy reproduces the e_exact stored by the optimization. This only
fixes the parameters; the reported energy is still the one measured with shots.
"""
import glob
import itertools
import json
import os
import re
import time

import numpy as np
from qiskit.quantum_info import Statevector

_KIND_PATTERN = re.compile(r"^(uccsd|uccgsd|kupccgsd)(\d*)(_tap)?$")
ALIGN_TOLERANCE = 1e-6   # Ha


def parse_kind(name: str):
    """'uccsd2_tap' -> ('uccsd', 2, True). Inverse of the naming used by vqe.sweep."""
    m = _KIND_PATTERN.match(name)
    if not m:
        raise ValueError(f"unknown ansatz name {name!r}")
    return m.group(1), int(m.group(2) or 1), m.group(3) is not None


def load_points(name: str, results_dir: str = "results") -> dict:
    """{d: {"theta", "e_exact"}} from Fede's results/<name>_d<d>.json files."""
    points = {}
    for path in glob.glob(os.path.join(results_dir, f"{name}_d*.json")):
        with open(path) as f:
            row = json.load(f)
        if row.get("kind") == name:
            points[float(row["d"])] = {"theta": list(row["theta"]), "e_exact": float(row["e_exact"])}
    return dict(sorted(points.items()))


def prepare_point(d: float, name: str):
    """(problem, ansatz) for distance d, exactly as vqe.sweep built them."""
    kind, k, taper = parse_kind(name)
    problem = build_problem(d)
    if taper:
        problem = taper_problem(problem)
    return problem, build_ansatz(problem, kind, k)


def _exact_energy(ansatz, theta, hamiltonian):
    return float(Statevector(ansatz.assign_parameters(theta)).expectation_value(hamiltonian).real)


def align_theta(problem: dict, ansatz, name: str, theta, e_exact: float):
    """Return (θ, sign pattern) such that the exact energy of the ansatz matches e_exact.

    Each parameter belongs to one excitation (the list repeats once per rep); its
    sign changes by the product of the signs of the spatial orbitals it touches.
    Raises if no pattern reproduces e_exact, which would mean something other than
    orbital signs changed.
    """
    kind, k, _ = parse_kind(name)
    theta = np.asarray(theta, dtype=float)
    excitations = excitation_list(problem, kind, k)
    if len(theta) != k * len(excitations):
        raise ValueError(f"θ has {len(theta)} values, expected {k} x {len(excitations)} excitations")
    n_orb = problem.get("num_spatial_orbitals", problem["n_qubits"] // 2)
    touched = [[q % n_orb for q in occ + virt] for occ, virt in excitations] * k

    best = None
    for signs in itertools.product((1, -1), repeat=n_orb):
        if signs[0] == -1:
            continue   # a global flip of all orbitals changes nothing; halve the search
        factor = np.array([np.prod([signs[p] for p in orbs]) for orbs in touched])
        candidate = theta * factor
        e = _exact_energy(ansatz, candidate, problem["hamiltonian"])
        if best is None or abs(e - e_exact) < abs(best[2] - e_exact):
            best = (candidate, signs, e)
        if abs(e - e_exact) < ALIGN_TOLERANCE:
            break
    if abs(best[2] - e_exact) > ALIGN_TOLERANCE:
        raise RuntimeError(f"d={problem['d']}: no orbital sign pattern reproduces e_exact "
                           f"({best[2]:.8f} vs {e_exact:.8f})")
    return best[0], best[1]


def measure_point(d: float, name: str, point: dict, total_shots: int, seed: int = 0,
                  reliability_shots: int = 0, allocation: str = "weighted") -> dict:
    """Measure one point of the curve and return its record (see analysis.make_record).

    `point` is {"theta", "e_exact"}. With reliability_shots > 0 it also measures
    <N>, <S²> and Var(H) (theory notes, section 11), stored under "reliability".
    """
    problem, ansatz = prepare_point(d, name)
    theta, signs = align_theta(problem, ansatz, name, point["theta"], point["e_exact"])
    t0 = time.time()
    m = measure_energy(ansatz, theta, problem["hamiltonian"], total_shots,
                       allocation=allocation, seed=seed)
    extra = {"n_qubits": problem["n_qubits"], "allocation": allocation,
             "e_exact": point["e_exact"], "orbital_signs": list(signs),
             "measure_seconds": round(time.time() - t0, 2)}
    if reliability_shots > 0:
        r = measure_reliability(ansatz, theta, problem["hamiltonian"], problem["number_op"],
                                problem["s2_op"], reliability_shots, allocation, seed=seed + 1)
        extra["reliability"] = {k: v for k, v in r.items() if k not in ("energy", "energy_sigma")}
    return make_record(name, problem, theta, m, count_cz(ansatz), **extra)


def measure_curve(name: str, points: dict, total_shots: int, seed: int = 0,
                  reliability_shots: int = 0, allocation: str = "weighted",
                  verbose: bool = True) -> list:
    """measure_point for every d in `points`; seeds seed, seed+10, ... per point."""
    records = []
    for i, (d, point) in enumerate(sorted(points.items())):
        t0 = time.time()
        rec = measure_point(d, name, point, total_shots, seed + 10 * i, reliability_shots, allocation)
        records.append(rec)
        if verbose:
            err = 1e3 * abs(rec["energy"] - rec["e_fci"])
            err_ccsd = 1e3 * abs(rec["e_ccsd"] - rec["e_fci"])
            z = (rec["energy"] - rec["e_exact"]) / rec["sigma"]
            print(f"d={d:.2f}: E={rec['energy']:.6f} ± {1e3 * rec['sigma']:.2f} mHa "
                  f"(vs exacta {z:+.1f}σ), |E-FCI|={err:.2f} mHa (CCSD {err_ccsd:.2f}), "
                  f"cz={rec['cz']}, {time.time() - t0:.1f} s", flush=True)
    return records


def reliability_table(records):
    """<N>, <S²> and Var(H) of every point that was measured with reliability_shots > 0."""
    import pandas as pd
    rows = []
    for r in sorted(records, key=lambda r: r["d"]):
        rel = r.get("reliability")
        if not rel:
            continue
        rows.append({"d (Å)": r["d"],
                     "<N>": rel["N"], "σ N": rel["N_sigma"],
                     "<S²>": rel["S2"], "σ S²": rel["S2_sigma"],
                     "Var(H) (Ha²)": rel["variance"], "σ Var": rel["variance_sigma"]})
    return pd.DataFrame(rows)

In [ ]:
# θ óptimos de uccsd_tap (optimización con statevector, generado desde results/uccsd_tap_d*.json)
FINAL_KIND = 'uccsd_tap'
POINTS = {
    0.6: {"e_exact": -1.9601877977805484, "theta": [0.004933079756, -0.001935437179, 0.004929536197, -0.001938733446, -0.018108309977, -0.038694530589, -0.031527574525, -0.019169133246, -0.013400435085, -0.013400379924, -0.098025922321, -0.031625832861, -0.019282933616, -0.018168043907]},
    0.7: {"e_exact": -2.1069896757978537, "theta": [0.005761230567, -0.002820252505, 0.005755260022, -0.002824520797, -0.021152763322, -0.045291568329, -0.041141848703, -0.025463771768, -0.019942448709, -0.019942390589, -0.118167311577, -0.041319227515, -0.024450070895, -0.02124830943]},
    0.8: {"e_exact": -2.167546132743841, "theta": [0.006296973367, -0.003681502935, 0.006287084559, -0.003686283496, -0.024225581834, -0.052063052518, -0.052286441671, -0.033145666919, -0.027955190918, -0.027954992022, -0.140302825886, -0.052588028968, -0.029964735498, -0.02437076323]},
    0.9: {"e_exact": -2.180281967086552, "theta": [0.006495398606, -0.004437506102, 0.006479948733, -0.004441917613, -0.027483361937, -0.058807452239, -0.065141169222, -0.042538241869, -0.037436533787, -0.037436595442, -0.164598730087, -0.065629543501, -0.03593118747, -0.027696170147]},
    1.0: {"e_exact": -2.1663080191850237, "theta": [0.006392454393, 0.00506317551, 0.006368943361, 0.005066211291, 0.031133772304, -0.065458004171, 0.079971442159, -0.054093117704, 0.048403045964, 0.048403219178, -0.191382879888, 0.080731276457, -0.042475713152, 0.031437446227]},
    1.1: {"e_exact": -2.1378051157937175, "theta": [0.006080978957, -0.005576333296, 0.00603895381, -0.005581337372, -0.035404872589, -0.07198953316, -0.09705061564, -0.068352990251, -0.060836241216, -0.060836693792, -0.220932226267, -0.098192812018, -0.049636078241, -0.035829781955]},
    1.2: {"e_exact": -2.102293901084378, "theta": [0.005685787366, 0.006037291769, 0.005614939227, 0.006047815735, 0.040535770441, -0.078326836295, 0.116561800906, -0.085872597004, 0.074585359454, 0.074586281761, -0.25324383393, 0.118217073511, -0.057299911909, 0.041121045734]},
    1.4: {"e_exact": -2.0281711489349123, "theta": [0.005305956176, 0.007193786257, 0.005118471068, 0.007244932218, 0.054326553714, -0.089670794673, 0.162716055735, -0.132285518619, 0.104449883456, 0.104453101844, -0.323993760106, 0.165768968131, -0.072920879642, 0.055359149358]},
    1.6: {"e_exact": -1.9656156024517355, "theta": [0.0060390426, 0.008932648735, 0.005702970694, 0.00902933766, 0.07345039529, -0.098316335404, 0.215928305724, -0.193355797438, 0.133801080827, 0.133803490274, -0.394741322232, 0.220423892546, -0.086700754771, 0.075027298319]},
    1.8: {"e_exact": -1.921142638312294, "theta": [0.006296357351, 0.009980434253, 0.006416920085, 0.0096476254, 0.095226120289, -0.106773102789, 0.270726410443, -0.262228559593, 0.160125011981, 0.160113765131, -0.452223000753, 0.275546120089, -0.099423529146, 0.097130718036]},
    2.0: {"e_exact": -1.8932755911297985, "theta": [0.003683307632, -0.009400692943, 0.006694153334, -0.006669789169, -0.114244114627, -0.120082818565, -0.321367078086, -0.328467983709, -0.184173688547, -0.184143269442, -0.489145135491, -0.324231094028, -0.115553293453, -0.115962847845]},
    2.2: {"e_exact": -1.8775240445796908, "theta": [-0.002321528844, -0.00839359835, 0.007864784701, 0.000339115274, -0.126731459927, -0.140460718383, -0.364383410371, -0.384236778437, -0.2069523123, -0.206926454066, -0.50836491024, -0.362977864516, -0.137741333688, -0.127862526748]},
    2.4: {"e_exact": -1.869441845861374, "theta": [-0.010726642283, 0.007897374747, 0.010582386359, -0.010082918922, 0.130959996295, -0.167851076122, 0.398734093229, -0.425820795478, 0.228556472911, 0.228579909286, -0.515381614682, 0.392186612192, -0.166426877776, 0.131381411261]},
    2.7: {"e_exact": -1.8649613322420813, "theta": [-0.029460823399, 0.011501961106, 0.021083929839, -0.030205477319, 0.117804618286, -0.222072785826, 0.437562283691, -0.459993920051, 0.257230849484, 0.257432021859, -0.509626694935, 0.422497153428, -0.222340800945, 0.117306066881]},
    3.0: {"e_exact": -1.8664592087793759, "theta": [-0.566915235757, 0.46250371188, 0.531239282263, -0.510144705062, 0.001249401545, -0.134105329489, 0.348559216846, -0.274767951296, 0.161959387622, 0.162583503844, -0.284619407629, 0.178487599096, -0.234239364682, 0.010865719092]},
}

In [ ]:
TOTAL_SHOTS = 5_000_000
RELIABILITY_SHOTS = 1_000_000

records = measure_curve(FINAL_KIND, POINTS, TOTAL_SHOTS, reliability_shots=RELIABILITY_SHOTS)
out = report(records)

In [ ]:
out["table"]

In [ ]:
out["figure"]

In [ ]:
out["summary"]

In [ ]:
check = out["beats_ccsd"]
print("le gana a CCSD en todos los puntos con d ≥ 1,8 Å:", check["all_ok"])
print("precisión química en esa zona:", check["all_chemical_accuracy"])
pd.DataFrame(check["points"])

In [ ]:
reliability_table(records)

In [ ]:
print(f"Tiempo total de ejecución: {(time.time() - _notebook_t0) / 60:.1f} min")